# Prepare SCI data
In this notebook, we load the final SCI scenario ensemble (v1.1) and select the emissions
variables needed for the rest of the pipeline.

Unlike the interim dataset used previously, this release is already harmonized to 2023
historical emissions and infilled centrally with a consistent methodology, so this
notebook no longer needs to run local harmonization/infilling (that used to happen in
`101_harmonise_infill_data.ipynb`, which is now retired). We just need to select the
relevant emissions variables and strip the SCI-specific naming wrapper back to the plain
`Emissions|<species>` convention the rest of the pipeline expects.

In [1]:
import pyam
import pandas as pd

import dotenv
import os
from pathlib import Path

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [2]:
dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [3]:
df = pyam.IamDataFrame(
    Path(
        os.environ['SCI_DATA']
    )
)

[INFO] 10:29:59 - pyam.core: Reading file /Users/hoegner/GitHub/ar7_netzero_assessment/inputs/SCI-2025_v1.1_prerelease_climate-assessment.xlsx


The `data` sheet contains all emissions variables, each wrapped as
`Climate Assessment|Harmonized and Infilled|Emissions|<species> [SCI v1.1]`, plus a few
precomputed MAGICC GSAT percentile summary rows (not needed here - we run the full
600-member ensemble ourselves later). We select just the emissions variables and strip the
wrapper back to plain `Emissions|<species>` names.

In [4]:
SCI_VARIABLE_PREFIX = "Climate Assessment|Harmonized and Infilled|"
SCI_VARIABLE_SUFFIX_MARKER = " [SCI"  # e.g. " [SCI v1.1]"


def strip_sci_wrapper(variable):
    """Strip the 'Climate Assessment|Harmonized and Infilled|...[SCI vX.X]' wrapper back
    to the plain 'Emissions|<species>' naming used throughout the rest of the pipeline
    (the same convention 101_harmonise_infill_data.ipynb used to produce as its output,
    before it was retired in favour of this centrally harmonized/infilled release)."""
    variable = variable.removeprefix(SCI_VARIABLE_PREFIX)
    if SCI_VARIABLE_SUFFIX_MARKER in variable:
        variable = variable.split(SCI_VARIABLE_SUFFIX_MARKER)[0]
    return variable


# Keep only the harmonized+infilled emissions variables (drops the precomputed GSAT
# percentile summary rows, which aren't needed for the full-ensemble MAGICC run below)
df_downselected = df.filter(variable=f"{SCI_VARIABLE_PREFIX}Emissions|*")

rename_map = {v: strip_sci_wrapper(v) for v in df_downselected.variable}
df_downselected.rename(variable=rename_map, inplace=True)

# Drop the SCI-precomputed Kyoto-gases aggregate. Investigated at length (see notes/
# for the full trail): it differs from our own in-house recomputation (ghg_mapping.yml
# + add_gwp100_kyoto, cross-checked against the real gcages.ghg_aggregation
# implementation) by a systematic ~2-3%, growing to double digits near net-zero years -
# regardless of which reasonable Kyoto-gas list is used (tested our own list, gcages's
# default list, and the emissions_harmonization_historical project's list - all agree
# with each other and all disagree with the SCI-precomputed value the same way). Root
# cause not resolved (likely a different GWP table or script version on the SCI
# production side; the file's variable name doesn't even match what the shown
# production code would produce). Decision: recompute in-house for full methodological
# control and reproducibility - 102_calculate_netzero_timings.ipynb does this whenever
# this variable isn't present in the input data.
df_downselected.filter(
    variable="Emissions|Kyoto Gases [AR6GWP100]",
    keep=False,
    inplace=True
)

df_downselected.variable

['Emissions|BC',
 'Emissions|C2F6',
 'Emissions|C3F8',
 'Emissions|C4F10',
 'Emissions|C5F12',
 'Emissions|C6F14',
 'Emissions|C7F16',
 'Emissions|C8F18',
 'Emissions|CCl4',
 'Emissions|CF4',
 'Emissions|CFC11',
 'Emissions|CFC113',
 'Emissions|CFC114',
 'Emissions|CFC115',
 'Emissions|CFC12',
 'Emissions|CH2Cl2',
 'Emissions|CH3Br',
 'Emissions|CH3CCl3',
 'Emissions|CH3Cl',
 'Emissions|CH4',
 'Emissions|CHCl3',
 'Emissions|CO',
 'Emissions|CO2',
 'Emissions|CO2|Biosphere',
 'Emissions|CO2|Fossil',
 'Emissions|HCFC141b',
 'Emissions|HCFC142b',
 'Emissions|HCFC22',
 'Emissions|HFC125',
 'Emissions|HFC134a',
 'Emissions|HFC143a',
 'Emissions|HFC152a',
 'Emissions|HFC227ea',
 'Emissions|HFC23',
 'Emissions|HFC236fa',
 'Emissions|HFC245fa',
 'Emissions|HFC32',
 'Emissions|HFC365mfc',
 'Emissions|HFC4310mee',
 'Emissions|Halon1202',
 'Emissions|Halon1211',
 'Emissions|Halon1301',
 'Emissions|Halon2402',
 'Emissions|N2O',
 'Emissions|NF3',
 'Emissions|NH3',
 'Emissions|NMVOC',
 'Emissions|NO

In [5]:
df_downselected.require_data(
    year=2100,
    exclude_on_fail=True
)

Ok, for some reason, it appears that the HFC32 go *negative* for some scenarios.
Get rid of those...

In [6]:
df_downselected.validate(
    variable='Emissions|HFC32',
    lower_bound=0,
    exclude_on_fail=True
)

In [7]:
df_downselected.filter(
    exclude=False,
    inplace=True
)

In [8]:
df_downselected.to_csv(
    OUTPUT_FOLDER / "100_harmonised_infilled_data.csv"
)